In [ ]:
import os
import sys

from setup import ProjectSetup


ImportError: cannot import name 'ProjectSetup' from 'setup' (/usr/local/lib/python3.12/dist-packages/setup/__init__.py)

In [4]:
setup      = ProjectSetup()
text_alice = setup.setup()

NameError: name 'ProjectSetup' is not defined

In [ ]:
from prepData import PrepareData

data = PrepareData(sequence_length=20, vocab_size=5000)

alice_path = os.path.join(setup.data_dir, "alice.txt")

if os.path.exists(alice_path):
    print("Alice déjà présent sur le Drive")
    text = data.load_from_file(alice_path)
else:
    print(" Téléchargement d'Alice...")
    text = data.load_from_url(
        url          = "https://www.gutenberg.org/files/11/11-0.txt",
        start_marker = "CHAPTER I.",
        end_marker   = "End of Project Gutenberg"
    )
    data.save_text(text, alice_path)

train_ds, val_ds = data.build(text)

In [ ]:
from SmallGPT import SmallGPT
from Model.trainer import Trainer

sequence_length = 20
embed_dim = 64
num_heads = 4
ff_dim = 128
num_layers = 2

model = SmallGPT(
    sequence_length = sequence_length,
    vocab_size  = data.vocab_size,
    embed_dim  = embed_dim,
    num_heads = num_heads,
    ff_dim  = ff_dim,
    num_layers = num_layers
)
model.compile(
    optimizer = "adam",
    loss      = "sparse_categorical_crossentropy",
    metrics   = ["accuracy"]
)


In [ ]:
trainer = Trainer(model, setup.model_path, epochs=60, patience=5)

if os.path.exists(setup.model_path):
    print(" reprise de l'entraînement")
    model = trainer.load()
else:
    print(" entraînement ")

history = trainer.train(train_ds, val_ds)